# Ταξινομηση σκυλου/γατας

Αυτο το notebook χρησιμοποιει το `data/oxford-iiit-pet/annotations/list.txt` για δυαδικες ετικετες ειδους. Ενα CNN μαθαινει να ταξινομει καθε πληρη εικονα ως γατα η σκυλος, και φορτωνεται το αποθηκευμενο checkpoint DeepLabv3+ ωστε η ιδια εικονα να μπορει επισης να τμηματοποιηθει.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from torchvision.transforms import InterpolationMode

from helper import DeepLabV3Plus, count_parameters, denormalize, set_seed

PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data" / "oxford-iiit-pet"
IMAGE_DIR = DATA_ROOT / "images"
ANNOTATION_DIR = DATA_ROOT / "annotations"
LIST_PATH = ANNOTATION_DIR / "list.txt"
DEEPLAB_CHECKPOINT = PROJECT_ROOT / "best_pet_segmentation_cnn.pt"
CLASSIFIER_CHECKPOINT = PROJECT_ROOT / "best_dog_cat_classifier_cnn.pt"

SEED = 42
IMAGE_SIZE = 160
BATCH_SIZE = 32
EPOCHS = 12
LEARNING_RATE = 1e-3
NUM_WORKERS = 0

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
SPECIES_NAMES = ["cat", "dog"]

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## Φορτωση δυαδικων ετικετων

Το `list.txt` εχει στηλες `image`, `class_id`, `species` και `breed_id`. Η στηλη species ειναι ηδη δυαδικη: το `1` σημαινει γατα και το `2` σημαινει σκυλος. Μετατρεπω αυτες τις τιμες σε `0` και `1`.

In [ ]:
def load_species_table(list_path=LIST_PATH):
    rows = []
    with list_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            image_id, class_id, species, breed_id = line.split()
            image_path = IMAGE_DIR / f"{image_id}.jpg"
            trimap_path = ANNOTATION_DIR / "trimaps" / f"{image_id}.png"
            if image_path.exists():
                species_id = int(species)
                rows.append(
                    {
                        "image_id": image_id,
                        "image_path": image_path,
                        "trimap_path": trimap_path,
                        "class_id": int(class_id),
                        "species": species_id,
                        "target": species_id - 1,
                        "label": SPECIES_NAMES[species_id - 1],
                        "breed_id": int(breed_id),
                    }
                )
    return pd.DataFrame(rows)


metadata = load_species_table()
print(metadata.shape)

class_distribution = metadata["label"].value_counts().rename_axis("label").to_frame("count")
class_distribution["fraction"] = class_distribution["count"] / class_distribution["count"].sum()
class_distribution

In [ ]:
def stratified_split(df, train_fraction=0.7, val_fraction=0.15, seed=SEED):
    rng = np.random.default_rng(seed)
    train_indices, val_indices, test_indices = [], [], []

    for _, group in df.groupby("target"):
        indices = group.index.to_numpy()
        rng.shuffle(indices)
        train_end = int(len(indices) * train_fraction)
        val_end = train_end + int(len(indices) * val_fraction)
        train_indices.extend(indices[:train_end])
        val_indices.extend(indices[train_end:val_end])
        test_indices.extend(indices[val_end:])

    return sorted(train_indices), sorted(val_indices), sorted(test_indices)


train_idx, val_idx, test_idx = stratified_split(metadata)
split_counts = pd.Series({"train": len(train_idx), "val": len(val_idx), "test": len(test_idx)})
split_counts

## Dataset και CNN ταξινομητης

Το dataset εχει περισσοτερες εικονες σκυλων απο εικονες γατων. Για αντισταθμιση, ο training loader χρησιμοποιει weighted sampler που δειγματοληπτει συχνοτερα παραδειγματα της μικροτερης κλασης. Οι validation και test loaders μενουν μη ισορροπημενοι, ωστε οι μετρικες να περιγραφουν ακομα τα πραγματικα holdout δεδομενα.

In [ ]:
train_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=InterpolationMode.BILINEAR),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ]
)

eval_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=InterpolationMode.BILINEAR),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ]
)


class PetSpeciesDataset(Dataset):
    def __init__(self, table, transform=None):
        self.table = table.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.table)

    def __getitem__(self, index):
        row = self.table.iloc[index]
        image = Image.open(row.image_path).convert("RGB")
        tensor = self.transform(image) if self.transform else transforms.ToTensor()(image)
        target = torch.tensor(row.target, dtype=torch.long)
        return tensor, target, row.image_id


train_dataset = PetSpeciesDataset(metadata.iloc[train_idx], transform=train_transform)
val_dataset = PetSpeciesDataset(metadata.iloc[val_idx], transform=eval_transform)
test_dataset = PetSpeciesDataset(metadata.iloc[test_idx], transform=eval_transform)

USE_WEIGHTED_SAMPLER = True
USE_CLASS_WEIGHTED_LOSS = False

train_targets = metadata.iloc[train_idx]["target"].to_numpy()
train_class_counts = np.bincount(train_targets, minlength=len(SPECIES_NAMES))
sample_weights = 1.0 / train_class_counts[train_targets]
train_sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)
class_weights = train_class_counts.sum() / (len(SPECIES_NAMES) * train_class_counts)
class_weights = torch.as_tensor(class_weights, dtype=torch.float32, device=device)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=not USE_WEIGHTED_SAMPLER,
    sampler=train_sampler if USE_WEIGHTED_SAMPLER else None,
    num_workers=NUM_WORKERS,
)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

pd.DataFrame(
    {
        "label": SPECIES_NAMES,
        "train_count": train_class_counts,
        "loss_weight": class_weights.detach().cpu().numpy(),
    }
)

In [ ]:
class DogCatCNN(nn.Module):
    def __init__(self, num_classes=2, dropout=0.35):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


classifier = DogCatCNN().to(device)
count_parameters(classifier)


## Εκπαιδευση του CNN σκυλου/γατας

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    correct = 0
    total = 0
    class_correct = torch.zeros(len(SPECIES_NAMES), dtype=torch.long)
    class_total = torch.zeros(len(SPECIES_NAMES), dtype=torch.long)

    for images, targets, _ in loader:
        images = images.to(device)
        targets = targets.to(device)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, targets)
            if training:
                loss.backward()
                optimizer.step()

        batch_size = targets.size(0)
        total_loss += loss.item() * batch_size
        predictions = logits.argmax(dim=1)
        correct += (predictions == targets).sum().item()
        total += batch_size

        for class_index in range(len(SPECIES_NAMES)):
            class_mask = targets.cpu() == class_index
            class_total[class_index] += class_mask.sum()
            class_correct[class_index] += (predictions.cpu()[class_mask] == class_index).sum()

    class_accuracy = class_correct.float() / class_total.clamp_min(1).float()
    metrics = {"loss": total_loss / total, "accuracy": correct / total}
    metrics["balanced_accuracy"] = class_accuracy.mean().item()
    for class_index, class_name in enumerate(SPECIES_NAMES):
        metrics[f"{class_name}_accuracy"] = class_accuracy[class_index].item()
    return metrics


criterion = nn.CrossEntropyLoss(weight=class_weights if USE_CLASS_WEIGHTED_LOSS else None)
optimizer = torch.optim.AdamW(classifier.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = []
best_val_balanced_accuracy = -1.0

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(classifier, train_loader, criterion, optimizer=optimizer)
    val_metrics = run_epoch(classifier, val_loader, criterion)
    scheduler.step()

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_balanced_accuracy": train_metrics["balanced_accuracy"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_balanced_accuracy": val_metrics["balanced_accuracy"],
        "val_cat_accuracy": val_metrics["cat_accuracy"],
        "val_dog_accuracy": val_metrics["dog_accuracy"],
    }
    history.append(row)
    print(row)

    if val_metrics["balanced_accuracy"] > best_val_balanced_accuracy:
        best_val_balanced_accuracy = val_metrics["balanced_accuracy"]
        torch.save(
            {
                "model_name": "DogCatCNN",
                "state_dict": classifier.state_dict(),
                "image_size": IMAGE_SIZE,
                "mean": MEAN,
                "std": STD,
                "class_names": SPECIES_NAMES,
                "validation_metrics": val_metrics,
                "imbalance_handling": {
                    "weighted_sampler": USE_WEIGHTED_SAMPLER,
                    "class_weighted_loss": USE_CLASS_WEIGHTED_LOSS,
                    "train_class_counts": train_class_counts.tolist(),
                    "class_weights": class_weights.detach().cpu().tolist(),
                },
                "seed": SEED,
            },
            CLASSIFIER_CHECKPOINT,
        )

history_df = pd.DataFrame(history)
history_df

In [ ]:
classifier_checkpoint = torch.load(CLASSIFIER_CHECKPOINT, map_location=device)
classifier.load_state_dict(classifier_checkpoint["state_dict"])
test_metrics = run_epoch(classifier, test_loader, criterion)
test_metrics

## Φορτωση του αποθηκευμενου segmenter DeepLabv3+

In [ ]:
# Το DeepLabV3Plus εισαγεται απο το helper.py.


In [ ]:
segmentation_checkpoint = torch.load(DEEPLAB_CHECKPOINT, map_location=device)
segmenter = DeepLabV3Plus(
    num_classes=len(segmentation_checkpoint["class_names"]),
    **segmentation_checkpoint["model_kwargs"],
).to(device)
segmenter.load_state_dict(segmentation_checkpoint["state_dict"])
segmenter.eval()

segmentation_transform = transforms.Compose(
    [
        transforms.Resize((segmentation_checkpoint["image_size"], segmentation_checkpoint["image_size"]), interpolation=InterpolationMode.BILINEAR),
        transforms.ToTensor(),
        transforms.Normalize(segmentation_checkpoint["mean"], segmentation_checkpoint["std"]),
    ]
)

segmentation_checkpoint["model_kwargs"], segmentation_checkpoint["test_metrics"]

## Ταξινομηση και τμηματοποιηση της ιδιας εικονας

In [ ]:
def predict_species(image):
    classifier.eval()
    tensor = eval_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        probabilities = classifier(tensor).softmax(dim=1).squeeze(0).cpu()
    predicted_index = int(probabilities.argmax())
    return SPECIES_NAMES[predicted_index], probabilities


def segment_image(image):
    original_size = image.size[::-1]
    tensor = segmentation_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = segmenter(tensor)
        prediction = logits.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
    prediction_image = Image.fromarray(prediction).resize(image.size, resample=Image.Resampling.NEAREST)
    return np.array(prediction_image)


def colorize_trimap(mask):
    colors = np.array(
        [
            [64, 180, 120],   # pet foreground
            [40, 80, 180],    # background
            [240, 190, 70],   # boundary
        ],
        dtype=np.uint8,
    )
    return colors[mask]


def overlay_mask(image, mask, alpha=0.45):
    image_array = np.array(image).astype(np.float32)
    mask_rgb = colorize_trimap(mask).astype(np.float32)
    return np.clip((1 - alpha) * image_array + alpha * mask_rgb, 0, 255).astype(np.uint8)


def classify_and_segment(image_id):
    row = metadata.loc[metadata["image_id"] == image_id].iloc[0]
    image = Image.open(row.image_path).convert("RGB")
    predicted_label, probabilities = predict_species(image)
    predicted_mask = segment_image(image)
    return {
        "image_id": image_id,
        "true_label": row.label,
        "predicted_label": predicted_label,
        "cat_probability": float(probabilities[0]),
        "dog_probability": float(probabilities[1]),
        "image": image,
        "mask": predicted_mask,
        "overlay": overlay_mask(image, predicted_mask),
    }


In [ ]:
sample_ids = metadata.iloc[test_idx].sample(6, random_state=SEED)["image_id"].tolist()
results = [classify_and_segment(image_id) for image_id in sample_ids]

fig, axes = plt.subplots(len(results), 3, figsize=(12, 4 * len(results)))
if len(results) == 1:
    axes = axes[None, :]

for row_axes, result in zip(axes, results):
    title = (
        f"{result['image_id']} | true: {result['true_label']} | "
        f"pred: {result['predicted_label']} "
        f"(cat={result['cat_probability']:.2f}, dog={result['dog_probability']:.2f})"
    )
    row_axes[0].imshow(result["image"])
    row_axes[0].set_title(title)
    row_axes[1].imshow(colorize_trimap(result["mask"]))
    row_axes[1].set_title("DeepLabv3+ segmentation")
    row_axes[2].imshow(result["overlay"])
    row_axes[2].set_title("Segmentation overlay")
    for ax in row_axes:
        ax.axis("off")

plt.tight_layout()